In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

In [70]:
import torchvision.datasets as datasets
mnist_trainset = datasets.MNIST(root=r'/home/xyphoes/Desktop/Projects/AIML - 2/Learning/Neural Networks/Projects/data', train=True, download=False, transform=None)
mnist_testset = datasets.MNIST(root=r'/home/xyphoes/Desktop/Projects/AIML - 2/Learning/Neural Networks/Projects/data', train=False, download=False, transform=None)

(xTrain, yTrain) = mnist_trainset.data.to(torch.float32), mnist_trainset.targets.to(torch.float32)
(xTest, yTest) = mnist_testset.data.to(torch.float32), mnist_testset.targets.to(torch.float32)
xTrain = xTrain / 255.0
xTest = xTest / 255.0
xTrain = xTrain.unsqueeze(1)  # shape: (N, 1, 28, 28)
xTest = xTest.unsqueeze(1)

In [71]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Since image grayscale, inchannel = 1. We want 6 feature maps, so outchannel=6. kernel size
        # is 5x5 hence 5.
        self.conv1 = nn.Conv2d(1, 6, 5)           # input: (batch, 1, 28, 28) -> (batch, 6, 24, 24)
        # This pooler pools separately for all 6 feature maps.
        self.pool = nn.MaxPool2d(2, 2)            # (batch, 6, 24, 24) -> (batch, 6, 12, 12)
        # This is the second convoluting layer producing 16 feature maps.
        self.conv2 = nn.Conv2d(6, 16, 5)          # (batch, 6, 12, 12) -> (batch, 16, 8, 8)
        # pool again: (batch, 16, 8, 8) -> (batch, 16, 4, 4)
        #fc1 is a linear layer that takes flattened input squeezes them into 120 neurons.
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.dropout = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # ReLU is applied after conv layers since convolution is linear. Relu introduces
        # non linearity.
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)  # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def predict_proba(self, X):
        logits = self.forward(X)
        probas = F.softmax(logits, 1)
        return probas

In [72]:
model = CNN()
optim = torch.optim.Adam(model.parameters(), lr = 0.01)
loss_fn = nn.CrossEntropyLoss()
n_epoch = 20

model.train()
for epoch in range(n_epoch):
    logits = model(xTrain)
    optim.zero_grad()
    loss = loss_fn(logits, yTrain.long())
    loss.backward()
    optim.step()
    print(f"Loss is: {round(float(loss), 3)} on epoch: {epoch}")
model.eval()

Loss is: 2.303 on epoch: 0
Loss is: 2.287 on epoch: 1
Loss is: 2.209 on epoch: 2
Loss is: 2.007 on epoch: 3
Loss is: 1.703 on epoch: 4
Loss is: 1.556 on epoch: 5
Loss is: 2.083 on epoch: 6
Loss is: 1.352 on epoch: 7
Loss is: 1.304 on epoch: 8
Loss is: 1.323 on epoch: 9
Loss is: 1.273 on epoch: 10
Loss is: 1.172 on epoch: 11
Loss is: 1.009 on epoch: 12
Loss is: 0.848 on epoch: 13
Loss is: 0.757 on epoch: 14
Loss is: 0.708 on epoch: 15
Loss is: 0.735 on epoch: 16
Loss is: 0.66 on epoch: 17
Loss is: 0.602 on epoch: 18
Loss is: 0.561 on epoch: 19


CNN(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)